# Grouping and Aggregating Data

Databricks free https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2672686114777595?o=7474657872577658#command/6613799583247549

# 01: Grouping and Aggregating Data with Spark

This demonstration will show how to perform grouping and aggregation operations using NYC Taxi trip data. We'll explore basic grouping, multiple aggregations, and window functions.

## Objectives

* Understand basic grouping operations in Spark
* Perform time-based analysis using aggregations
* Implement complex aggregations with multiple metrics
* Use window functions for advanced analytics
* Optimize aggregation performance

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that Serverless is enabled by default.

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

* In the drop-down, select **More**.
* In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
2. Find the triangle icon to the right of your compute cluster name and click it.
3. Wait a few minutes for the cluster to start.
4. Once the cluster is running, complete the steps above to select your cluster.

## A. Data Setup and Loading

First, let's load our taxi trip data and examine its structure.

In [0]:
%python
from pyspark.sql.functions import *

# Read and displaying the taxi data
#samples.nyctaxi.trips
trips_df = spark.read.table("samples.nyctaxi.trips")

display(trips_df.limit(10))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


## B. Basic Grouping Operations

Let's start with simple grouping operations to understand trip patterns by location.

In [0]:
from pyspark.sql.functions import desc

# Contar viagens por local de recolha para mostrar os 5 mais populares
location_counts = trips_df \
    .groupBy("pickup_zip") \
    .count() \
    .orderBy(desc("count"))

display(location_counts.limit(5))

pickup_zip,count
10001,1227
10003,1181
10011,1129
10021,1021
10018,1012


In [0]:
display(location_counts.show(15))

+----------+-----+
|pickup_zip|count|
+----------+-----+
|     10001| 1227|
|     10003| 1181|
|     10011| 1129|
|     10021| 1021|
|     10018| 1012|
|     10023| 1008|
|     10028|  929|
|     10012|  834|
|     10110|  763|
|     10065|  702|
|     10017|  694|
|     10119|  675|
|     10009|  661|
|     10002|  613|
|     10103|  583|
+----------+-----+
only showing top 15 rows


In [0]:
%sql
SELECT pickup_zip, COUNT(*) AS count
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY count DESC
LIMIT 5;

pickup_zip,count
10001,1227
10003,1181
10011,1129
10021,1021
10018,1012


# C. Combining Multiple Aggregations

In [0]:
from pyspark.sql.functions import count, avg, sum, round, desc

# Perform multiple aggregations by location, order by most popular pickup locations
location_stats = trips_df \
    .groupBy("pickup_zip") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(sum("fare_amount"), 2).alias("total_fare_amt")
    ) \
    .orderBy(desc("total_trips"))

display(location_stats.limit(5))

pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt
10001,1227,2.21,10.62,13028.01
10003,1181,2.32,10.98,12965.5
10011,1129,2.29,10.91,12321.5
10021,1021,2.02,10.21,10424.0
10018,1012,2.6,11.4,11541.51


In [0]:
%sql
SELECT 
    pickup_zip,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(SUM(fare_amount), 2) AS total_fare_amt
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY total_trips DESC
LIMIT 5;

pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt
10001,1227,2.21,10.62,13028.01
10003,1181,2.32,10.98,12965.5
10011,1129,2.29,10.91,12321.5
10021,1021,2.02,10.21,10424.0
10018,1012,2.6,11.4,11541.51


# D. Window Functions

Diferença entre ntile() e funções de ranking (rank(), dense_rank(), row_number())
```
    • rank(), dense_rank(), row_number(): Atribuem uma posição ou nota a cada linha baseada em seu valor (ex: 1º lugar, 2º lugar, 3º lugar).
    • ntile(n): Não mede "quem está em primeiro ou segundo lugar" no sentido de pontuação pura; ela categoriza as linhas em "fatias" ou "segmentos" de volume equivalente.
 ```
Diferença rápida: NTILE vs RANK
```
    • NTILE(3): Ignora se os valores são muito distantes ou próximos; ele foca puramente em criar 3 grupos com a mesma quantidade de linhas (ex: 33% dos dados em cada).
    • RANK(): Vai dar o mesmo número de ranking para valores idênticos e pular posições (ex: 1, 2, 2, 4), sem se importar em criar fatias de tamanhos iguais.
```



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, ntile, desc

# Create window specs for different ranking methods
window_by_trips = Window.orderBy(desc("total_trips"))
window_by_fare = Window.orderBy(desc("avg_fare"))

# Add different types of rankings
ranked_locations = location_stats \
    .withColumn("trips_rank", rank().over(window_by_trips)) \
    .withColumn("fare_rank", rank().over(window_by_fare)) \
    .withColumn("fare_quintile", ntile(5).over(window_by_fare))  # Divide into 5 groups by fare

display(ranked_locations.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt,trips_rank,fare_rank,fare_quintile
8876,1,0.0,260.0,260.0,101,1,1
7974,1,0.0,188.0,188.0,101,2,1
7114,1,0.0,105.0,105.0,101,3,1
7310,1,0.0,105.0,105.0,101,3,1
7311,1,2.0,60.0,60.0,101,5,1


In [0]:
ranked_locations.select('fare_quintile').distinct().show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+
|fare_quintile|
+-------------+
|            2|
|            4|
|            3|
|            5|
|            1|
+-------------+



In [0]:
from pyspark.sql.functions import max

ranked_locations.select(max('fare_rank')).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+
|max(fare_rank)|
+--------------+
|           127|
+--------------+



In [0]:
from pyspark.sql.functions import min

ranked_locations.select(min('fare_rank')).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+
|min(fare_rank)|
+--------------+
|             1|
+--------------+



In [0]:
ranked_locations.select('fare_rank').distinct().show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+
|fare_rank|
+---------+
|       37|
|       27|
|       71|
|      100|
|      120|
|       88|
|       80|
|       94|
|      104|
|       47|
|       46|
|       83|
|       55|
|       28|
|      107|
|       74|
|        2|
|       73|
|       64|
|       95|
+---------+
only showing top 20 rows


In [0]:
%sql
WITH location_stats AS (
SELECT 
    pickup_zip,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(SUM(fare_amount), 2) AS total_fare_amt
FROM samples.nyctaxi.trips
GROUP BY pickup_zip)
, ranked_locations AS (
    SELECT 
        *,
        RANK() OVER (ORDER BY total_trips DESC) AS trips_rank,
        RANK() OVER (ORDER BY avg_fare DESC) AS fare_rank,
        NTILE(5) OVER (ORDER BY avg_fare DESC) AS fare_quintile
    FROM location_stats
)
SELECT * 
FROM ranked_locations 
LIMIT 5;

pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt,trips_rank,fare_rank,fare_quintile
10001,1227,2.21,10.62,13028.01,1,99,4
10003,1181,2.32,10.98,12965.5,2,86,4
10011,1129,2.29,10.91,12321.5,3,91,4
10021,1021,2.02,10.21,10424.0,4,104,5
10018,1012,2.6,11.4,11541.51,5,79,4


In [0]:
%sql
-- ==========================================
-- 1. Agrupamiento Básico (Basic Grouping)
-- ==========================================
SELECT pickup_zip, COUNT(*) AS count
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY count DESC
LIMIT 5;

-- ==========================================
-- 2. Agregaciones Múltiples (Multiple Aggregations)
-- ==========================================
CREATE OR REPLACE TEMP VIEW location_stats AS
SELECT 
    pickup_zip,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(SUM(fare_amount), 2) AS total_fare_amt
FROM samples.nyctaxi.trips
GROUP BY pickup_zip;

-- Visualizar as estatísticas agregadas principais
SELECT * 
FROM location_stats
ORDER BY total_trips DESC
LIMIT 5;

-- ==========================================
-- 3. Funciones de Ventana (Window Functions)
-- ==========================================
WITH ranked_locations AS (
    SELECT 
        *,
        RANK() OVER (ORDER BY total_trips DESC) AS trips_rank,
        RANK() OVER (ORDER BY avg_fare DESC) AS fare_rank,
        NTILE(5) OVER (ORDER BY avg_fare DESC) AS fare_quintile
    FROM location_stats
)
SELECT * 
FROM ranked_locations 
ORDER BY total_trips DESC
LIMIT 5;

pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt,trips_rank,fare_rank,fare_quintile
10001,1227,2.21,10.62,13028.01,1,99,4
10003,1181,2.32,10.98,12965.5,2,86,4
10011,1129,2.29,10.91,12321.5,3,91,4
10021,1021,2.02,10.21,10424.0,4,104,5
10018,1012,2.6,11.4,11541.51,5,79,4


In [0]:
# Displaying the results
display(ranked_locations.select(
    "pickup_zip",
    "total_trips",
    "avg_fare",
    "avg_distance",
    "trips_rank",
    "fare_rank",
    "fare_quintile"
).limit(15))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


pickup_zip,total_trips,avg_fare,avg_distance,trips_rank,fare_rank,fare_quintile
8876,1,260.0,0.0,101,1,1
7974,1,188.0,0.0,101,2,1
7114,1,105.0,0.0,101,3,1
7310,1,105.0,0.0,101,3,1
7311,1,60.0,2.0,101,5,1
11430,4,52.0,17.28,79,6,1
11368,1,52.0,8.9,101,6,1
11420,1,52.0,14.8,101,6,1
11422,429,44.78,15.5,24,9,1
11436,11,42.32,12.96,73,10,1


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# ==========================================================================
# E. Particionamento (Partitioning)
# ==========================================================================

# 1) Window.partitionBy() — a função de janela é calculada DENTRO de cada grupo
#    (partição lógica), e não na tabela inteira. Aqui: cada pickup_zip é
#    ranqueado DENTRO do seu quintil de tarifa. Isso também elimina o warning
#    "No Partition Defined for Window operation", que aparece quando a janela
#    usa apenas orderBy (todos os dados vão para uma única partição).
window_by_quintile = Window.partitionBy("fare_quintile").orderBy(desc("total_trips"))

partitioned_locations = ranked_locations.withColumn(
    "rank_in_quintile",  # posição do zip code dentro do seu quintil
    row_number().over(window_by_quintile)
)

display(
    partitioned_locations.select(
        "pickup_zip", "total_trips", "avg_fare",
        "fare_quintile", "rank_in_quintile"
    ).orderBy("fare_quintile", "rank_in_quintile").limit(15)
)

# 2) repartition() — particionamento FÍSICO dos dados no cluster: linhas com a
#    mesma chave vão para a mesma partição, reduzindo shuffles em agrupamentos
#    e joins subsequentes. Diferente do Window.partitionBy, que é lógico/análitico.
location_stats_repartitioned = location_stats.repartition("pickup_zip")

location_stats_repartitioned.groupBy("pickup_zip").count().orderBy(desc("count")).show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


pickup_zip,total_trips,avg_fare,fare_quintile,rank_in_quintile
11371,483,30.64,1,1
11422,429,44.78,1,2
11370,26,25.31,1,3
11436,11,42.32,1,4
11369,5,24.2,1,5
11430,4,52.0,1,6
11221,4,21.75,1,7
11375,3,33.0,1,8
11230,3,20.83,1,9
11224,3,20.33,1,10


+----------+-----+
|pickup_zip|count|
+----------+-----+
|     10103|    1|
|     10002|    1|
|     10022|    1|
|     10280|    1|
|     10278|    1|
+----------+-----+
only showing top 5 rows


# F. Particionamento em SQL (PARTITION BY)

### `PARTITION BY` em SQL — o equivalente do `Window.partitionBy()` (cell 25)

Dentro do `OVER(...)`, o `PARTITION BY` divide as linhas em grupos e calcula a função de janela **dentro de cada grupo**, de forma independente:

* `NTILE(5) OVER (ORDER BY avg_fare DESC)` → sem `PARTITION BY`: o quintil é calculado na tabela inteira
* `RANK() OVER (PARTITION BY fare_quintile ORDER BY total_trips DESC)` → cada zip é ranqueado **dentro** do seu quintil
* `AVG(...) OVER (PARTITION BY ...)` → agregações de janela também aceitam `PARTITION BY`, sem precisar de `GROUP BY`

**Nota:** um alias criado num `SELECT` não pode ser referenciado no mesmo nível da instrução — por isso usamos CTEs encadeadas (`location_stats` → `ranked_locations` → `partitioned`).

In [0]:
%sql
-- ============================================================
-- F. Particionamento em SQL: PARTITION BY dentro do OVER()
-- Equivalente ao Window.partitionBy() da cell 25
-- ============================================================
WITH location_stats AS (
    SELECT
        pickup_zip,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(SUM(fare_amount), 2) AS total_fare_amt
    FROM samples.nyctaxi.trips
    GROUP BY pickup_zip
),
ranked_locations AS (
    SELECT
        *,
        NTILE(5) OVER (ORDER BY avg_fare DESC) AS fare_quintile  -- sem partição: tabela inteira
    FROM location_stats
),
partitioned AS (
    SELECT
        pickup_zip,
        total_trips,
        avg_fare,
        fare_quintile,
        -- PARTITION BY: o rank reinicia dentro de cada quintil
        RANK() OVER (
            PARTITION BY fare_quintile
            ORDER BY total_trips DESC
        ) AS rank_in_quintile,
        -- agregação de janela: média de viagens por quintil, sem GROUP BY
        ROUND(AVG(total_trips) OVER (PARTITION BY fare_quintile), 1) AS avg_trips_in_quintile
    FROM ranked_locations
)
SELECT *
FROM partitioned
ORDER BY fare_quintile, rank_in_quintile
LIMIT 20;

pickup_zip,total_trips,avg_fare,fare_quintile,rank_in_quintile,avg_trips_in_quintile
11371,483,30.64,1,1,38.2
11422,429,44.78,1,2,38.2
11370,26,25.31,1,3,38.2
11436,11,42.32,1,4,38.2
11369,5,24.2,1,5,38.2
11430,4,52.0,1,6,38.2
11221,4,21.75,1,6,38.2
11230,3,20.83,1,8,38.2
11224,3,20.33,1,8,38.2
11375,3,33.0,1,8,38.2


# having 

In [0]:
from pyspark.sql.functions import col, count, avg, round, desc

# ==========================================================================
# G. HAVING — filtrar DEPOIS da agregação
#    (WHERE filtra linhas ANTES do GROUP BY; HAVING filtra GRUPOS depois)
# ==========================================================================

# ------------------------------------------------------------------
# 1) Verificar dados duplicados com HAVING
#    Agrupa pelas colunas-chave: grupos com COUNT(*) > 1 são duplicados
# ------------------------------------------------------------------
demonstration_trips = spark.createDataFrame([
    ("2026-11-01 08:00:00", 10001, 10068, 2.5, 12.50),
    ("2026-11-01 08:00:00", 10001, 10068, 2.5, 12.50),  # duplicado
    ("2026-11-01 08:00:00", 10001, 10068, 2.5, 12.50),  # duplicado
    ("2026-11-01 09:30:00", 10022, 10103, 5.0, 22.00),
    ("2026-11-01 10:00:00", 10022, 10103, 1.2,  8.75),
    ("2026-11-01 10:00:00", 10022, 10103, 1.2,  8.75),   # duplicado
], ["pickup_datetime", "pickup_zip", "dropoff_zip", "trip_distance", "fare_amount"])

demonstration_trips.createOrReplaceTempView("demonstration_trips")

In [0]:
demonstration_trips.show()

+-------------------+----------+-----------+-------------+-----------+
|    pickup_datetime|pickup_zip|dropoff_zip|trip_distance|fare_amount|
+-------------------+----------+-----------+-------------+-----------+
|2026-11-01 08:00:00|     10001|      10068|          2.5|       12.5|
|2026-11-01 08:00:00|     10001|      10068|          2.5|       12.5|
|2026-11-01 08:00:00|     10001|      10068|          2.5|       12.5|
|2026-11-01 09:30:00|     10022|      10103|          5.0|       22.0|
|2026-11-01 10:00:00|     10022|      10103|          1.2|       8.75|
|2026-11-01 10:00:00|     10022|      10103|          1.2|       8.75|
+-------------------+----------+-----------+-------------+-----------+



In [0]:
# a) SQL clássico: GROUP BY nas colunas-chave + HAVING COUNT(*) > 1
spark.sql("""
    SELECT pickup_datetime, pickup_zip, dropoff_zip,
           trip_distance, fare_amount,
           COUNT(*) AS ocorrencias
    FROM demonstration_trips
    GROUP BY pickup_datetime, pickup_zip, dropoff_zip, trip_distance, fare_amount
    HAVING COUNT(*) > 1
    ORDER BY ocorrencias DESC
""").show()


+-------------------+----------+-----------+-------------+-----------+-----------+
|    pickup_datetime|pickup_zip|dropoff_zip|trip_distance|fare_amount|ocorrencias|
+-------------------+----------+-----------+-------------+-----------+-----------+
|2026-11-01 08:00:00|     10001|      10068|          2.5|       12.5|          3|
|2026-11-01 10:00:00|     10022|      10103|          1.2|       8.75|          2|
+-------------------+----------+-----------+-------------+-----------+-----------+



In [0]:

# b) Equivalente PySpark: .filter() DEPOIS do groupBy() é o "HAVING" da API
demonstration_trips \
    .groupBy("pickup_datetime", "pickup_zip", "dropoff_zip",
             "trip_distance", "fare_amount") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-------------------+----------+-----------+-------------+-----------+-----+
|    pickup_datetime|pickup_zip|dropoff_zip|trip_distance|fare_amount|count|
+-------------------+----------+-----------+-------------+-----------+-----+
|2026-11-01 08:00:00|     10001|      10068|          2.5|       12.5|    3|
|2026-11-01 10:00:00|     10022|      10103|          1.2|       8.75|    2|
+-------------------+----------+-----------+-------------+-----------+-----+



In [0]:
# ------------------------------------------------------------------
# 2) HAVING sobre os dados reais: filtrar AGREGADOS
#    total_trips e avg_fare só existem DEPOIS do GROUP BY —
#    um WHERE não conseguiria filtrar por eles
# ------------------------------------------------------------------
spark.sql("""
    SELECT pickup_zip,
           COUNT(*)                    AS total_trips,
           ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM samples.nyctaxi.trips
    GROUP BY pickup_zip
    HAVING COUNT(*) > 100 AND AVG(fare_amount) > 25
    ORDER BY total_trips DESC
""").show()

+----------+-----------+--------+
|pickup_zip|total_trips|avg_fare|
+----------+-----------+--------+
|     11371|        483|   30.64|
|     11422|        429|   44.78|
+----------+-----------+--------+



In [0]:

# Equivalente PySpark DataFrame
trips_df \
    .groupBy("pickup_zip") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    ) \
    .filter((col("total_trips") > 100) & (col("avg_fare") > 25)) \
    .orderBy(desc("total_trips")) \
    .show()

+----------+-----------+--------+
|pickup_zip|total_trips|avg_fare|
+----------+-----------+--------+
|     11371|        483|   30.64|
|     11422|        429|   44.78|
+----------+-----------+--------+



In [0]:

# ------------------------------------------------------------------
# 3) Verificar duplicados nos dados REAIS (samples.nyctaxi.trips)
#    Resultado vazio = não existem duplicados nessa chave de negócio
# ------------------------------------------------------------------
spark.sql("""
    SELECT tpep_pickup_datetime, tpep_dropoff_datetime,
           pickup_zip, dropoff_zip, trip_distance, fare_amount,
           COUNT(*) AS ocorrencias
    FROM samples.nyctaxi.trips
    GROUP BY tpep_pickup_datetime, tpep_dropoff_datetime,
             pickup_zip, dropoff_zip, trip_distance, fare_amount
    HAVING COUNT(*) > 1
    LIMIT 10
""").show()

+--------------------+---------------------+----------+-----------+-------------+-----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_zip|dropoff_zip|trip_distance|fare_amount|ocorrencias|
+--------------------+---------------------+----------+-----------+-------------+-----------+-----------+
+--------------------+---------------------+----------+-----------+-------------+-----------+-----------+

